# Cost, Latency, and Agent Economics

## Scenario: make the EU checkout investigation economically reliable

A user request can cause model calls, tool fan-out, searches, retries, evaluators, and escalation. We will turn an unbounded trajectory into a budgeted one that uses cache, a fast route, bounded parallel reads, and reasoning only for uncertainty.

**Safety boundary:** budgets limit work; they do not authorize tools. The simulated system only prepares a proposal.

![Agent economics budget and decision lifecycle](../../../assets/agent-economics.svg)

The controller accounts for actual tokens, tool calls, spend, and wall-clock time after every stage. When a quality floor is not met, it can promote once only if capacity remains; otherwise it returns an evidence-backed limitation or requests review.

## 1. Budget the complete trajectory

Set independent limits for context/tokens, model/reasoning turns, action/tool calls, retries, spend, and end-to-end latency. Include router, queue, cache, tool, evaluator, and review overhead. The objective is not token minimization: it is the least expensive route that meets a measured quality floor, policy, and p95 latency target.

A small/fast model fits a tested stable path. A reasoning model earns its cost only for ambiguity or constraint trade-offs. Multimodal and coding paths are capability requirements. Cache hits are valid only when tenant, authorization, source freshness, and policy/catalog version all match.

In [1]:
from pathlib import Path
import sys
TOPIC = Path.cwd()
if not (TOPIC / 'lab.py').exists():
    TOPIC = Path.cwd() / 'curriculum' / 'enterprise-agent' / '07-cost-latency-agent-economics'
sys.path.insert(0, str(TOPIC))
from lab import Budget, Trace, charge, investigate

cached = investigate(cache_hit=True, uncertain=False)
complex_case = investigate(cache_hit=False, uncertain=True)
for name, trace in [('cached', cached), ('complex', complex_case)]:
    print(name, trace.events, 'latency=', trace.latency_ms, 'ms cost=', trace.spend_cents, 'cents')
assert cached.latency_ms == 30
assert complex_case.events[-1] == 'run:reasoning-synthesis'

cached ['run:cache'] latency= 30 ms cost= 0.01 cents
complex ['run:fast-classify', 'run:parallel-status-and-incidents', 'run:reasoning-synthesis'] latency= 3450 ms cost= 1.65 cents


## 2. Parallelism, caching, and sequential gates

Parallel read-only calls can lower wall-clock latency, but they still spend money and can overload an upstream service. Add concurrency limits, cancellation, individual timeouts, and a shared deadline. Keep dependent work sequential: a cheap classifier or eligibility check can avoid a costly search. Treat cache output as untrusted until its scope/freshness/provenance checks pass.

Speculative execution launches a likely next read before the branch is final. Use it only when expected latency benefit exceeds expected wasted cost, cancellation works, and the input is authorized. Never speculate writes or high-risk customer/production operations.

In [2]:
parallel = investigate(cache_hit=False, uncertain=False, parallel_reads=True)
sequential = investigate(cache_hit=False, uncertain=False, parallel_reads=False)
print('parallel wall-clock:', parallel.latency_ms, 'ms; sequential:', sequential.latency_ms, 'ms')
print('same tool spend:', parallel.spend_cents, sequential.spend_cents)
assert parallel.spend_cents == sequential.spend_cents
assert parallel.latency_ms < sequential.latency_ms

# Deliberate failure: no budget remains for an expensive recovery route.
tight, trace = Budget(tokens=100, actions=1, spend_cents=.1, latency_ms=100), Trace()
assert not charge(trace, tight, 'reasoning-retry', 300, 1, .2, 500)
print(trace.events)

parallel wall-clock: 1600 ms; sequential: 2200 ms
same tool spend: 0.38 0.38
['stop:reasoning-retry:budget-exceeded']


## 3. Dynamic routing, promotions, and economics evaluation

Use hard capability/policy filters first, then rank eligible routes by held-out task-family quality, expected cost, time remaining, and current availability. A promotion must have an external acceptance signal—schema/test/evidence validation, calibrated evaluator, or review—not only model self-confidence. Record catalog and policy version, selected route, cache state, promotion/fallback reason, predicted/actual metrics, and final quality.

Evaluate outcome and trajectory: task/evidence/policy success; cost per successful safe task; p50/p95/p99 latency; cache and fallback hit rate; retries; queue time; route-quality calibration; and quality/cost slices by risk, language, modality, and tenant. Regressions after a price, provider, prompt, tool, or policy change should block release.

## Production checklist and exercises

- Allocate per-run, tenant, workflow, and global circuit-breaker budgets; reserve capacity for a safe stop or escalation.
- Apply idempotency, backoff/jitter, and retry classification to actions; every retry consumes budget.
- Keep authorization, approval, identity, and tenant controls outside the model/router.
- Test timeout, cache poisoning/staleness, model outage, fan-out overload, cascade exhaustion, and degraded quality.

**Exercises:** add an idempotency-required write retry; model expected value for speculative search; implement a one-promotion quality gate with a hard deadline; and define a dashboard that finds low-cost routes with poor success.

References: [FrugalGPT](https://arxiv.org/abs/2305.05176), [RouteLLM](https://arxiv.org/abs/2406.18665), [RouterBench](https://arxiv.org/abs/2403.12031), [Unified routing and cascading](https://arxiv.org/abs/2410.10347).